In [1]:
# Import required libraries
import os
import pandas as pd

In [2]:
# Load the dataset
df = pd.read_csv('../data/train_transcriptomics_SDY2941.tsv', sep='\t')
df = df.drop(columns=['transcriptomics_id'])
df.head()

,participant_id,timepoint,ensembl_gene_id,raw_count,tpm_count,material
0,SDY2941.SUB395006,7,ENSG00000000003,14.0,0.197000,Whole blood
1,SDY2941.SUB395006,3,ENSG00000000003,3.0,0.049508,Whole blood
2,SDY2941.SUB395006,0,ENSG00000000003,1.0,0.013695,Whole blood
3,SDY2941.SUB395005,90,ENSG00000000003,5.0,0.073354,Whole blood
4,SDY2941.SUB395005,90,ENSG00000000003,2.0,0.030201,Whole blood


In [3]:
# raw_count has actual values (no nulls); material is uniformly 'Whole blood'
# SDY2941 has 6 timepoints: [0, 3, 7, 28, 85, 90]
# Drop material before pivoting; drop raw_count to keep tpm_count only, consistent with 2019/2020 pipeline
print('raw_count null fraction:', df['raw_count'].isna().mean())
print('material unique values:', df['material'].unique())
print('timepoints:', sorted(df['timepoint'].unique()))
print('participants:', df['participant_id'].nunique())

raw_count null fraction: 0.0
material unique values: <StringArray>
['Whole blood']
Length: 1, dtype: str
timepoints: [np.int64(0), np.int64(3), np.int64(7), np.int64(28), np.int64(85), np.int64(90)]
participants: 53


In [4]:
# Drop material (entirely 'Whole blood'); drop raw_count to match pipeline (tpm_count only)
df = df.drop(columns=['raw_count', 'material'])
df.head()

,participant_id,timepoint,ensembl_gene_id,tpm_count
0,SDY2941.SUB395006,7,ENSG00000000003,0.197000
1,SDY2941.SUB395006,3,ENSG00000000003,0.049508
2,SDY2941.SUB395006,0,ENSG00000000003,0.013695
3,SDY2941.SUB395005,90,ENSG00000000003,0.073354
4,SDY2941.SUB395005,90,ENSG00000000003,0.030201


In [5]:
# Pivot: each row is a (participant_id, timepoint), each gene becomes a column
df_pivot = df.pivot_table(
    index=['participant_id', 'timepoint'],
    columns='ensembl_gene_id',
    values='tpm_count'
)
df_pivot.columns.name = None
df_pivot = df_pivot.reset_index()
print(df_pivot.shape)
df_pivot.head()

(316, 54904)


,participant_id,timepoint,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,...,ENSG00000310527,ENSG00000310529,ENSG00000310530,ENSG00000310532,ENSG00000310533,ENSG00000310534,ENSG00000310535,ENSG00000310536,ENSG00000310537,ENSG00000310539
0,SDY2941.SUB394970,0,0.076436,0.0,28.472615,7.858272,1.775265,511.323918,1.137430,8.304955,...,49.349637,0.0,0.0,0.0,4.274253,0.000000,5.606868,0.532530,2.564265,0.840145
1,SDY2941.SUB394970,3,0.063310,0.0,26.014101,5.642286,1.146896,421.281203,1.544501,10.750257,...,44.654943,0.0,0.0,0.0,4.037608,2.073928,3.202315,1.111025,3.367633,1.158509
2,SDY2941.SUB394970,7,0.093166,0.0,23.904538,6.154335,1.068919,334.987489,1.510813,11.026327,...,58.108749,0.0,0.0,0.0,4.324523,0.736028,9.540562,0.699061,2.467432,0.487317
3,SDY2941.SUB394970,28,0.050954,0.0,28.014155,7.250600,0.894216,386.884501,1.259193,9.393705,...,58.187897,0.0,0.0,0.0,5.660007,1.207308,4.076846,0.322652,3.427343,0.966716
4,SDY2941.SUB394970,85,0.135118,0.0,26.334418,6.022043,1.680267,382.058179,0.993043,9.156669,...,54.641882,0.0,0.0,0.0,4.362678,1.366445,5.039315,0.905511,1.958800,2.518913


In [7]:
# Save the cleaned dataset to the cleaned_data folder
os.makedirs('../cleaned_data', exist_ok=True)
df_pivot.to_csv('../cleaned_data/transcriptomics_SDY2941_cleaned.csv', index=False)